# HW08: Information Retrieval & RAG
## CIS 5300: Natural Language Processing | Module 10 | 100 Points

---

### Overview

In this assignment, you will build a complete **information retrieval** and **retrieval-augmented generation (RAG)** pipeline from scratch. You will:

1. Implement **sparse retrieval** using TF-IDF and BM25
2. Implement **dense retrieval** using pretrained sentence embeddings
3. Build a **RAG pipeline** that retrieves relevant passages and uses an LLM to generate answers
4. **Evaluate** both retrieval quality and generation faithfulness
5. **Reflect** on the design tradeoffs in RAG systems

This homework connects directly to Module 10 lectures (M10.1.1 through M10.4.3) and Chapter 11 of the textbook.

### Learning Goals

After completing this homework, you will be able to:

1. **Implement TF-IDF and BM25** from scratch and understand how term weighting works *(M10.1.1, Chapter 11 Section 11.1)*
2. **Use dense retrieval** with bi-encoder models and compare to sparse methods *(M10.2.1, Chapter 11 Section 11.3)*
3. **Build a RAG pipeline** end-to-end: chunk, index, retrieve, and generate *(M10.4.1-M10.4.3, Chapter 11 Section 11.4)*
4. **Evaluate retrieval** using precision@k and recall@k *(Chapter 11, IR evaluation)*
5. **Analyze RAG tradeoffs** including chunking strategies and when RAG helps vs. hurts *(M10.4.2-M10.4.3)*

### Deliverables

Submit the following files to **Gradescope** as a single submission:

| File | Description |
|------|-------------|
| `hw08.py` | **Required.** Export from this notebook (*File → Download → Download .py*). The autograder imports it to test your Section 1–3 functions. |
| `hw08.ipynb` | This notebook with all code cells completed and outputs visible |
| `hw08_writeup.pdf` | Written answers to all questions marked **[WRITEUP]** |

### Point Breakdown

| Section | Points | Type |
|---------|--------|------|
| 1. Sparse Retrieval (TF-IDF + BM25) | 20 | Code (autograded) |
| 2. Dense Retrieval | 20 | Code + Writeup |
| 3. Building a RAG Pipeline | 30 | Code + TA review |
| 4. RAG Evaluation & Analysis | 18 | Code + Writeup |
| 5. Reflection | 12 | Writeup |
| **Total** | **100** | |

### Recommended Reading

- [Chapter 11: Information Retrieval and RAG](https://web.stanford.edu/~jurafsky/slp3/11.pdf) — Jurafsky & Martin, SLP 3rd Edition
- Module 10 Lectures: M10.1.1 through M10.4.3

### API Key Setup

Section 3 uses an **LLM API** for generation — **Anthropic (default) or OpenAI**. You only need **one account** for the whole course. Set the matching key as an environment variable before running the notebook:

```bash
export ANTHROPIC_API_KEY=os.environ.get("API_KEY", "")
```

Or set it in the notebook (not recommended for shared notebooks):
```python
import os
os.environ["ANTHROPIC_API_KEY"] = "your-key-here"
```

**Estimated API cost:** ~$0.50 for the entire homework (about 6 API calls at temperature=0).

**Per-problem labels:** `[auto-checked]` = the autograder verifies your function via imports (Sections 1–3 deterministic code); `[manually graded]` = a TA grades against the rubric with no fixed autograder points (e.g. 2.2, whose code just produces numbers you analyze in the writeup); `[WRITEUP — manually graded]` = the graded artifact is your written answer in `hw08_writeup.pdf`; 3.3 uses a one-off label, `[structure auto-checked · 15 pts by TA from notebook output]`, since the autograder only checks structure while a TA assigns all 15 points from the notebook output. Final section scores are assigned by TAs.

---
## Section 0: Setup

Run the following cells to install dependencies and load the data. **Do not modify these cells.**

In [ ]:
# Install dependencies (guarded so the autograder can `import` this file
# without triggering a pip install; the notebook still installs on Run All).
if __name__ == '__main__':
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "numpy", "sentence-transformers", "anthropic", "openai"])

In [ ]:
import numpy as np
import math
import re
import os
from collections import Counter, defaultdict
from typing import List, Dict, Tuple

In [ ]:
if __name__ == '__main__':
    ########## DO NOT MODIFY ##########
    # Document corpus: A collection of passages on various NLP and AI topics.
    # Each document has an 'id', 'title', and 'text' field.

    DOCUMENTS = [
        {"id": 0, "title": "Transformer Architecture",
         "text": "The Transformer architecture was introduced in the paper Attention Is All You Need by Vaswani et al. in 2017. It relies entirely on self-attention mechanisms, dispensing with recurrence and convolutions. The model uses multi-head attention to jointly attend to information from different representation subspaces. Transformers have become the foundation for most modern NLP models including BERT, GPT, and T5."},
        {"id": 1, "title": "BERT Pre-training",
         "text": "BERT (Bidirectional Encoder Representations from Transformers) was developed by Google in 2018. It is pre-trained using two objectives: masked language modeling (MLM), where random tokens are masked and the model predicts them, and next sentence prediction (NSP), where the model predicts whether two sentences are consecutive. BERT uses a bidirectional transformer encoder, allowing it to consider context from both directions."},
        {"id": 2, "title": "Word Embeddings",
         "text": "Word embeddings represent words as dense vectors in a continuous space. Word2Vec, introduced by Mikolov et al. in 2013, learns embeddings using either the Skip-gram or CBOW architecture. GloVe, developed at Stanford, learns embeddings from co-occurrence statistics. These embeddings capture semantic relationships: for example, the vector for king minus man plus woman approximately equals the vector for queen."},
        {"id": 3, "title": "Attention Mechanisms",
         "text": "Attention mechanisms allow neural networks to focus on relevant parts of the input when producing output. The basic attention mechanism computes a weighted sum of value vectors, where the weights are determined by the compatibility between a query and key vectors. Scaled dot-product attention divides the dot product by the square root of the key dimension. Multi-head attention runs multiple attention operations in parallel."},
        {"id": 4, "title": "Sentiment Analysis",
         "text": "Sentiment analysis is the task of determining the emotional tone or opinion expressed in text. It can be performed at the document level, sentence level, or aspect level. Common approaches include lexicon-based methods using sentiment dictionaries, machine learning classifiers like Naive Bayes and logistic regression, and deep learning models like BERT fine-tuned for sentiment classification."},
        {"id": 5, "title": "Machine Translation History",
         "text": "Machine translation has evolved from rule-based systems in the 1950s to statistical methods in the 1990s to neural approaches today. Statistical machine translation used phrase-based models and language models trained on parallel corpora. Neural machine translation, introduced around 2014, uses encoder-decoder architectures with attention. The transformer architecture dramatically improved translation quality starting in 2017."},
        {"id": 6, "title": "Language Model Training",
         "text": "Large language models are trained on massive text corpora using self-supervised learning. The training objective is typically next-token prediction: given a sequence of tokens, predict the next token. Models like GPT use autoregressive training, processing text left to right. Training requires enormous computational resources, often thousands of GPUs running for weeks. The resulting models can generate coherent text and perform various NLP tasks."},
        {"id": 7, "title": "Named Entity Recognition",
         "text": "Named Entity Recognition (NER) identifies and classifies named entities in text into categories such as person names, organizations, locations, dates, and monetary values. Modern NER systems use sequence labeling with BIO tagging. Pre-trained language models like BERT have significantly improved NER performance. NER is a fundamental component of information extraction pipelines."},
        {"id": 8, "title": "Text Classification Methods",
         "text": "Text classification assigns predefined categories to text documents. Traditional approaches use bag-of-words features with classifiers like Naive Bayes, SVM, or logistic regression. Feature engineering includes TF-IDF weighting, n-grams, and part-of-speech tags. Modern approaches fine-tune pre-trained language models on labeled data. Common applications include spam detection, topic categorization, and sentiment analysis."},
        {"id": 9, "title": "Information Retrieval Basics",
         "text": "Information retrieval is the task of finding relevant documents from a large collection given a user query. The classic approach uses the vector space model with TF-IDF term weighting and cosine similarity. An inverted index maps terms to the documents containing them for efficient lookup. Modern retrieval systems combine traditional keyword-based methods with neural dense retrieval approaches."},
        {"id": 10, "title": "Tokenization Methods",
         "text": "Tokenization breaks text into smaller units called tokens. Word-level tokenization splits on whitespace and punctuation. Subword tokenization methods like BPE (Byte Pair Encoding) and WordPiece handle unknown words by breaking them into known subword units. SentencePiece provides a language-independent tokenization framework. The choice of tokenizer significantly affects model performance."},
        {"id": 11, "title": "Retrieval-Augmented Generation",
         "text": "Retrieval-augmented generation (RAG) combines information retrieval with language model generation. Given a query, the system first retrieves relevant documents from a corpus, then conditions the language model on these documents to generate an answer. RAG reduces hallucination by grounding responses in retrieved evidence. It also enables answering questions about private or recent data not in the model's training set."},
        {"id": 12, "title": "Dense Passage Retrieval",
         "text": "Dense passage retrieval (DPR) uses neural networks to encode queries and passages into dense vector representations. A bi-encoder architecture encodes queries and passages separately, enabling efficient retrieval through approximate nearest neighbor search. DPR overcomes the vocabulary mismatch problem of sparse retrieval by capturing semantic similarity. Training requires positive and negative passage pairs for contrastive learning."},
        {"id": 13, "title": "BM25 Algorithm",
         "text": "BM25 is a ranking function used in information retrieval to score documents against a query. It extends TF-IDF by adding term frequency saturation and document length normalization. The formula includes parameters k1 (controlling term frequency scaling) and b (controlling length normalization). With typical values k1=1.5 and b=0.75, BM25 consistently outperforms basic TF-IDF. BM25 remains competitive with neural methods for keyword-heavy queries."},
        {"id": 14, "title": "Question Answering Systems",
         "text": "Question answering systems automatically answer questions posed in natural language. Extractive QA identifies answer spans within given passages. Generative QA uses language models to generate answers from scratch or conditioned on context. Modern QA systems often use a retriever-reader architecture: the retriever finds relevant passages and the reader extracts or generates answers from them."},
        {"id": 15, "title": "Cosine Similarity",
         "text": "Cosine similarity measures the cosine of the angle between two vectors. It ranges from -1 (opposite) to 1 (identical direction), with 0 indicating orthogonality. In information retrieval, cosine similarity is used to compare query and document vectors. It is length-invariant, meaning it only considers the direction of vectors, not their magnitude. This makes it suitable for comparing documents of different lengths."},
        {"id": 16, "title": "Hallucination in Language Models",
         "text": "Hallucination occurs when language models generate text that is fluent and plausible but factually incorrect. LLMs may fabricate facts, cite non-existent sources, or produce inconsistent information. Hallucination is especially problematic in high-stakes domains like medicine and law. Retrieval-augmented generation helps reduce hallucination by grounding outputs in retrieved documents. However, models may still hallucinate even with relevant context provided."},
        {"id": 17, "title": "Vector Databases",
         "text": "Vector databases are specialized systems for storing and querying high-dimensional vector embeddings. Popular options include Faiss, Pinecone, Weaviate, and Chroma. They support approximate nearest neighbor search algorithms like HNSW and IVF for efficient similarity queries. Vector databases are essential infrastructure for dense retrieval and RAG systems operating at scale with millions of documents."},
        {"id": 18, "title": "Prompt Engineering",
         "text": "Prompt engineering is the practice of designing input prompts to elicit desired behavior from language models. Techniques include zero-shot prompting, few-shot prompting with examples, chain-of-thought reasoning, and instruction following. The format and wording of prompts can significantly affect model outputs. In RAG systems, prompt engineering determines how retrieved context is presented to the language model."},
        {"id": 19, "title": "Chunking Strategies for RAG",
         "text": "Document chunking splits long documents into smaller segments for indexing in RAG systems. Fixed-size chunking divides text every N tokens. Sentence-based chunking respects sentence boundaries. Semantic chunking uses embeddings to find natural topic breaks. Overlapping chunks share tokens between adjacent segments to preserve context at boundaries. Chunk size involves tradeoffs: smaller chunks are more precise but lose context, while larger chunks preserve context but may include irrelevant information."},
        {"id": 20, "title": "Evaluation Metrics for IR",
         "text": "Information retrieval systems are evaluated using metrics that account for ranking. Precision at k (P@k) measures the fraction of relevant documents in the top k results. Recall at k (R@k) measures the fraction of all relevant documents found in the top k. Mean Average Precision (MAP) averages precision at each relevant document position. Mean Reciprocal Rank (MRR) uses the reciprocal of the rank of the first relevant result."},
        {"id": 21, "title": "Transfer Learning in NLP",
         "text": "Transfer learning adapts models trained on one task to perform well on another. In NLP, this typically involves pre-training a language model on large text corpora, then fine-tuning on a specific downstream task. BERT popularized this approach for NLP in 2018. Pre-training captures general language understanding, while fine-tuning specializes the model. This approach dramatically reduces the amount of labeled data needed for new tasks."},
        {"id": 22, "title": "Semantic Search",
         "text": "Semantic search goes beyond keyword matching to understand the meaning and intent behind queries. It uses dense vector representations to capture semantic similarity between queries and documents. A query about automobile problems can match documents about car issues even without shared keywords. Semantic search typically uses bi-encoder models to embed queries and documents into a shared vector space for similarity comparison."},
        {"id": 23, "title": "TF-IDF Weighting",
         "text": "TF-IDF (Term Frequency-Inverse Document Frequency) is a numerical statistic reflecting the importance of a word to a document in a collection. Term frequency measures how often a word appears in a document, typically log-scaled. Inverse document frequency measures how rare a word is across the collection. Words appearing in few documents have high IDF and are more discriminative. The TF-IDF score is the product of these two components."},
        {"id": 24, "title": "Neural Network Fundamentals",
         "text": "Neural networks are composed of layers of interconnected nodes that transform input data through learned weights and nonlinear activation functions. Feedforward networks pass information in one direction. Recurrent networks maintain hidden states for sequential data. Convolutional networks use local filters for spatial patterns. Training uses backpropagation and gradient descent to minimize a loss function."},
        {"id": 25, "title": "RLHF and Alignment",
         "text": "Reinforcement Learning from Human Feedback (RLHF) aligns language models with human preferences. The process involves training a reward model on human comparisons of model outputs, then using reinforcement learning to optimize the language model against this reward. RLHF helps models produce more helpful, harmless, and honest responses. Alternative approaches include Direct Preference Optimization (DPO) and Constitutional AI."},
        {"id": 26, "title": "Document Indexing",
         "text": "Document indexing creates data structures for efficient retrieval. An inverted index maps terms to lists of documents containing them, along with term frequencies and positions. For dense retrieval, document embeddings are computed offline and stored in vector indices. Hybrid systems maintain both sparse (inverted index) and dense (vector) indices. The indexing pipeline includes document parsing, chunking, embedding generation, and index construction."},
        {"id": 27, "title": "Few-Shot Learning",
         "text": "Few-shot learning enables models to perform tasks with very few labeled examples. In-context learning provides examples in the prompt without updating model weights. GPT-3 demonstrated strong few-shot performance across many tasks. Meta-learning approaches train models to learn quickly from few examples. Few-shot methods are particularly valuable when labeled data is expensive or scarce."},
        {"id": 28, "title": "Text Preprocessing",
         "text": "Text preprocessing transforms raw text into a format suitable for analysis. Common steps include lowercasing, removing punctuation, tokenization, stop word removal, and stemming or lemmatization. For information retrieval, preprocessing affects both index construction and query processing. Modern neural models often use minimal preprocessing, relying on subword tokenizers like BPE to handle the full vocabulary."},
        {"id": 29, "title": "Bi-Encoder Architecture",
         "text": "The bi-encoder architecture uses two separate encoders to independently encode queries and documents into dense vectors. At index time, all documents are encoded and their vectors stored. At query time, only the query is encoded, and similarity is computed via dot product with stored document vectors. This architecture is far more efficient than cross-encoders, which must jointly encode each query-document pair. Bi-encoders trade some accuracy for dramatic efficiency gains."},
        {"id": 30, "title": "Cross-Encoder Reranking",
         "text": "Cross-encoders jointly encode a query-document pair using a single transformer, allowing fine-grained token-level interactions between query and document. This produces more accurate relevance scores than bi-encoders but is too expensive for full collection search. In practice, cross-encoders are used to rerank the top results from an efficient first-stage retriever like BM25 or a bi-encoder. This two-stage approach combines efficiency with accuracy."},
        {"id": 31, "title": "Knowledge Graphs",
         "text": "Knowledge graphs represent structured information as entities and relationships. Examples include Wikidata, Freebase, and Google's Knowledge Graph. They store facts as triples: subject-predicate-object (e.g., Paris-capitalOf-France). Knowledge graphs can augment language models by providing structured facts for question answering. Graph neural networks can reason over knowledge graph structures."},
        {"id": 32, "title": "Evaluation of Language Models",
         "text": "Language models are evaluated using perplexity, which measures how well the model predicts held-out text. Lower perplexity indicates better prediction. For downstream tasks, evaluation metrics include accuracy, F1 score, BLEU for translation, and ROUGE for summarization. Human evaluation assesses fluency, coherence, and factual accuracy. Benchmarks like MMLU, HellaSwag, and TruthfulQA test different capabilities."},
        {"id": 33, "title": "Data Augmentation for NLP",
         "text": "Data augmentation creates additional training examples to improve model robustness. Techniques include synonym replacement, random insertion and deletion, back-translation, and paraphrasing with language models. For text classification, augmentation can address class imbalance. Augmented data should preserve the original label while introducing useful variation. Over-augmentation can introduce noise and degrade performance."},
        {"id": 34, "title": "Vocabulary Mismatch Problem",
         "text": "The vocabulary mismatch problem occurs when queries and relevant documents use different words to express the same concept. For example, a query about automobile repair might not match a document about car maintenance because the terms differ. Sparse retrieval methods like TF-IDF and BM25 rely on exact word overlap and cannot bridge this gap. Dense retrieval using learned embeddings addresses this by mapping semantically similar texts to nearby vectors."},
        {"id": 35, "title": "Contextual Embeddings",
         "text": "Contextual embeddings produce different vector representations for the same word depending on its context. Unlike static embeddings (Word2Vec, GloVe) that assign one vector per word, models like ELMo and BERT generate context-dependent representations. The word bank has different embeddings in river bank versus bank account. Contextual embeddings significantly improve performance on tasks requiring word sense disambiguation."},
        {"id": 36, "title": "RAG Prompt Construction",
         "text": "In retrieval-augmented generation, the prompt provided to the language model includes retrieved passages as context. A typical RAG prompt contains: an instruction to answer based on provided documents, the retrieved passages (often numbered with source information), and the user's question. Effective RAG prompts instruct the model to cite sources and acknowledge when information is insufficient. Document ordering in the prompt can affect answer quality due to the lost-in-the-middle phenomenon."},
        {"id": 37, "title": "Contrastive Learning for Retrieval",
         "text": "Contrastive learning trains retrieval models by pulling positive pairs closer and pushing negative pairs apart in embedding space. For dense retrieval, positive pairs are (query, relevant passage) and negative pairs are (query, irrelevant passage). Hard negatives — passages that are superficially similar but not relevant — are more informative for training than random negatives. InfoNCE loss and triplet loss are common training objectives."},
        {"id": 38, "title": "Multi-hop Question Answering",
         "text": "Multi-hop question answering requires reasoning across multiple documents or passages to find an answer. For example, answering What is the capital of the country where the Eiffel Tower is located requires first identifying France, then finding its capital. Multi-hop QA systems decompose complex questions into sub-questions, retrieve evidence for each, and synthesize a final answer. Datasets like HotpotQA benchmark this capability."},
        {"id": 39, "title": "Sparse vs Dense Retrieval",
         "text": "Sparse retrieval methods like BM25 represent documents as high-dimensional sparse vectors based on word frequencies. Dense retrieval methods use neural networks to create low-dimensional dense vectors capturing semantic meaning. Sparse methods excel at exact keyword matching and are fast and interpretable. Dense methods handle synonyms and paraphrases but require neural computation. Hybrid approaches combining both often achieve the best results in practice."}
    ]

    # Ground-truth relevance judgments for evaluation
    # Format: {query_id: {"query": str, "relevant_doc_ids": [int]}}
    EVAL_QUERIES = {
        0: {"query": "How does the transformer model work?",
            "relevant_doc_ids": [0, 3]},
        1: {"query": "What is BERT and how is it trained?",
            "relevant_doc_ids": [1, 21]},
        2: {"query": "How does TF-IDF weighting work in information retrieval?",
            "relevant_doc_ids": [9, 23, 15]},
        3: {"query": "What is retrieval-augmented generation?",
            "relevant_doc_ids": [11, 36, 14]},
        4: {"query": "How do dense retrieval systems work?",
            "relevant_doc_ids": [12, 29, 22]},
        5: {"query": "What is the vocabulary mismatch problem?",
            "relevant_doc_ids": [34, 22, 39]},
        6: {"query": "How does BM25 scoring work?",
            "relevant_doc_ids": [13, 9, 23]},
        7: {"query": "What causes hallucination in language models?",
            "relevant_doc_ids": [16, 11]},
        8: {"query": "How are documents chunked for RAG systems?",
            "relevant_doc_ids": [19, 26]},
        9: {"query": "How do you evaluate information retrieval systems?",
            "relevant_doc_ids": [20, 9]},
        10: {"query": "What are the tradeoffs between bi-encoders and cross-encoders?",
             "relevant_doc_ids": [29, 30, 12]},
        11: {"query": "How do automobile repair guides help car owners?",
             "relevant_doc_ids": [34]},
    }

    # Questions for RAG evaluation (Section 3-4)
    RAG_QUESTIONS = [
        "What is retrieval-augmented generation and why does it help with hallucination?",
        "How does BM25 differ from basic TF-IDF?",
        "What are the advantages of bi-encoder architecture over cross-encoders for retrieval?",
        "What chunking strategies can be used in RAG systems and what are their tradeoffs?",
        "What is the vocabulary mismatch problem and how does dense retrieval address it?",
    ]

    print(f"Corpus: {len(DOCUMENTS)} documents")
    print(f"Evaluation queries: {len(EVAL_QUERIES)}")
    print(f"RAG questions: {len(RAG_QUESTIONS)}")
    print(f"\nExample document:")
    print(f"  Title: {DOCUMENTS[0]['title']}")
    print(f"  Text: {DOCUMENTS[0]['text'][:100]}...")

---
## Section 1: Sparse Retrieval (20 points)

In this section, you will implement two classic sparse retrieval methods: **TF-IDF** and **BM25**. Both represent queries and documents as sparse vectors based on word frequencies, and score documents using cosine similarity or the BM25 formula.

Refer to **Lecture M10.1.1** and **Chapter 11, Section 11.1** for the mathematical details.

### 1.0 Text Preprocessing (provided)

We provide a simple preprocessing function. All retrieval methods should use this to tokenize text.

In [ ]:
########## DO NOT MODIFY ##########
from typing import List   # so this cell runs even if you skipped the imports cell
import re

def preprocess(text: str) -> List[str]:
    """Tokenize and lowercase text, removing punctuation.

    Args:
        text: input string

    Returns:
        List of lowercase tokens
    """
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    tokens = text.split()
    return tokens

if __name__ == '__main__':
    # Test preprocessing
    print(preprocess("The Transformer architecture was introduced in 2017!"))

### 1.1 TF-IDF Retrieval (10 points) [auto-checked]

Implement TF-IDF retrieval from scratch. You need to:

1. **Compute TF**: $\text{tf}_{t,d} = 1 + \log_{10}(\text{count}(t,d))$ if count > 0, else 0
2. **Compute IDF**: $\text{idf}_t = \log_{10}(N / \text{df}_t)$ where $N$ is the number of documents and $\text{df}_t$ is the number of documents containing term $t$
3. **Score**: $\text{score}(q,d) = \sum_{t \in q} \text{tf-idf}(t,q) \cdot \text{tf-idf}(t,d)$ (with L2 normalization)

This is the cosine similarity between TF-IDF vectors — see **Chapter 11, Equation 11.9**.

**How the class is organized (read this before coding):**
- **`__init__` builds the index once:** compute `df`/`idf` over the whole corpus, then pre-compute and store each document's TF-IDF vector and its L2 norm. Do all four `## YOUR CODE HERE` steps here.
- **The helper methods are reused by both indexing and querying:** `_compute_tf` and `_compute_tfidf_vector` turn a token list into a vector (used for documents in `__init__` and for the query in `retrieve`); `_cosine_similarity` scores one query vector against one doc vector.
- **`retrieve` builds the query's TF-IDF vector, then scores it against every stored doc vector** and returns the top-k `(doc_id, score)` pairs. Here `doc_id` is simply the document's index in the list (0..N-1).
- **Out-of-vocabulary query terms:** a query word not seen in the corpus has no IDF. Treat its IDF as **0** (use `self.idf.get(term, 0)`, not `self.idf[term]`) so it contributes nothing — otherwise you'll hit a `KeyError` on words like "does".

In [ ]:
class TFIDFRetriever:
    """TF-IDF retriever using cosine similarity.

    Implements the formulas from Chapter 11, Section 11.1:
    - TF: 1 + log10(count) if count > 0, else 0
    - IDF: log10(N / df)
    - Score: cosine similarity of TF-IDF vectors
    """

    def __init__(self, documents: List[Dict]):
        """Build the index from a list of documents.

        Args:
            documents: list of dicts with 'id', 'title', 'text' keys
        """
        self.documents = documents
        self.N = len(documents)

        # Preprocess and store tokenized documents
        self.doc_tokens = []
        for doc in documents:
            tokens = preprocess(doc['title'] + ' ' + doc['text'])
            self.doc_tokens.append(tokens)

        ## YOUR CODE HERE
        # 1. Compute document frequency (df) for each term:
        #    df[term] = number of documents containing term
        # 2. Compute IDF for each term:
        #    idf[term] = log10(N / df[term])
        # 3. Compute and store TF-IDF vectors for each document
        #    (as dicts mapping term -> tf-idf weight)
        # 4. Compute and store L2 norms for each document vector
        #    (for cosine similarity normalization)

        self.df = {}  # term -> document frequency
        self.idf = {}  # term -> inverse document frequency
        self.doc_vectors = []  # list of dicts: term -> tf-idf weight
        self.doc_norms = []  # list of floats: L2 norm of each doc vector

        raise NotImplementedError("Implement TFIDFRetriever.__init__ (build tf-idf vectors and norms) before running this cell.")

    def _compute_tf(self, tokens: List[str]) -> Dict[str, float]:
        """Compute log-scaled term frequency for a token list.

        Formula: tf = 1 + log10(count) if count > 0, else 0

        Args:
            tokens: list of preprocessed tokens

        Returns:
            Dict mapping term -> tf value
        """
        ## YOUR CODE HERE
        raise NotImplementedError("Implement TFIDFRetriever._compute_tf before running this cell.")

    def _compute_tfidf_vector(self, tokens: List[str]) -> Dict[str, float]:
        """Compute TF-IDF vector for a token list.

        Args:
            tokens: list of preprocessed tokens

        Returns:
            Dict mapping term -> tf-idf weight
        """
        ## YOUR CODE HERE
        raise NotImplementedError("Implement TFIDFRetriever._compute_tfidf_vector before running this cell.")

    def _cosine_similarity(self, vec1: Dict[str, float], norm1: float,
                           vec2: Dict[str, float], norm2: float) -> float:
        """Compute cosine similarity between two sparse vectors.

        Args:
            vec1, vec2: sparse vectors as dicts (term -> weight)
            norm1, norm2: precomputed L2 norms

        Returns:
            Cosine similarity (float). Returns 0.0 if either norm is 0.
        """
        ## YOUR CODE HERE
        raise NotImplementedError("Implement TFIDFRetriever._cosine_similarity before running this cell.")

    def retrieve(self, query: str, top_k: int = 5) -> List[Tuple[int, float]]:
        """Retrieve top-k documents for a query.

        Args:
            query: query string
            top_k: number of documents to return

        Returns:
            List of (doc_id, score) tuples, sorted by score descending
        """
        ## YOUR CODE HERE
        raise NotImplementedError("Implement TFIDFRetriever.retrieve before running this cell.")

In [ ]:
if __name__ == '__main__':
    # Test TF-IDF retriever
    tfidf_retriever = TFIDFRetriever(DOCUMENTS)

    # Sanity check: query about TF-IDF should return the TF-IDF document
    results = tfidf_retriever.retrieve("TF-IDF term weighting", top_k=5)
    print("Query: 'TF-IDF term weighting'")
    for doc_id, score in results:
        print(f"  [{doc_id}] {DOCUMENTS[doc_id]['title']}: {score:.4f}")

    # The TF-IDF document (id=23) should be in the top results
    top_ids = [doc_id for doc_id, _ in results]
    assert 23 in top_ids, f"Expected doc 23 (TF-IDF Weighting) in results, got {top_ids}"
    print("\nSanity check passed!")

### 1.2 BM25 Retrieval (5 points) [auto-checked]

Implement BM25 scoring. The BM25 formula (see **Chapter 11, Equation 11.12**) is:

$$\text{score}(q, d) = \sum_{t \in q} \text{idf}(t) \cdot \frac{\text{tf}_{t,d}}{k_1 \left(1 - b + b \cdot \frac{|d|}{d_{\text{avg}}}\right) + \text{tf}_{t,d}}$$

where:
- $\text{idf}(t) = \log_{10}(N / \text{df}_t)$
- $\text{tf}_{t,d}$ is the **raw** count of term $t$ in document $d$ (NOT log-scaled)
- $k_1$ controls term frequency saturation (default 1.5)
- $b$ controls length normalization (default 0.75)
- $|d|$ is the document length in tokens
- $d_{\text{avg}}$ is the average document length

**Note:** As in 1.1, skip out-of-vocabulary query terms (a term not in `self.idf`) instead of indexing `self.idf[term]` directly.

In [ ]:
class BM25Retriever:
    """BM25 retriever.

    Implements BM25 scoring from Chapter 11, Equation 11.12.
    Uses IDF as log10(N / df_t) and raw term counts (not log-scaled).
    """

    def __init__(self, documents: List[Dict], k1: float = 1.5, b: float = 0.75):
        """Build the BM25 index.

        Args:
            documents: list of dicts with 'id', 'title', 'text' keys
            k1: term frequency saturation parameter (default 1.5)
            b: document length normalization parameter (default 0.75)
        """
        self.documents = documents
        self.N = len(documents)
        self.k1 = k1
        self.b = b

        # Preprocess documents
        self.doc_tokens = []
        for doc in documents:
            tokens = preprocess(doc['title'] + ' ' + doc['text'])
            self.doc_tokens.append(tokens)

        ## YOUR CODE HERE
        # 1. Compute document lengths and average document length
        # 2. Compute document frequency (df) for each term
        # 3. Compute IDF for each term: log10(N / df)
        # 4. Store raw term counts for each document

        raise NotImplementedError("Implement BM25Retriever.__init__ before running this cell.")

    def retrieve(self, query: str, top_k: int = 5) -> List[Tuple[int, float]]:
        """Retrieve top-k documents for a query using BM25 scoring.

        Args:
            query: query string
            top_k: number of documents to return

        Returns:
            List of (doc_id, score) tuples, sorted by score descending
        """
        ## YOUR CODE HERE
        raise NotImplementedError("Implement BM25Retriever.retrieve before running this cell.")

In [ ]:
if __name__ == '__main__':
    # Test BM25 retriever
    bm25_retriever = BM25Retriever(DOCUMENTS)

    results = bm25_retriever.retrieve("BM25 ranking algorithm", top_k=5)
    print("Query: 'BM25 ranking algorithm'")
    for doc_id, score in results:
        print(f"  [{doc_id}] {DOCUMENTS[doc_id]['title']}: {score:.4f}")

    # BM25 document (id=13) should be in top results
    top_ids = [doc_id for doc_id, _ in results]
    assert 13 in top_ids, f"Expected doc 13 (BM25 Algorithm) in results, got {top_ids}"
    print("\nSanity check passed!")

### 1.3 Evaluate Sparse Retrieval (5 points) [auto-checked]

Implement **precision@k** and **recall@k** (see **Chapter 11** on IR evaluation, and **Lecture M10.1.1**), then evaluate both TF-IDF and BM25 on the provided evaluation queries.

$$\text{Precision@k} = \frac{|\{\text{relevant docs in top-k}\}|}{k}$$

$$\text{Recall@k} = \frac{|\{\text{relevant docs in top-k}\}|}{|\{\text{all relevant docs}\}|}$$

In [ ]:
def precision_at_k(retrieved_ids: List[int], relevant_ids: List[int], k: int) -> float:
    """Compute precision@k.

    Args:
        retrieved_ids: list of retrieved document IDs (in ranked order)
        relevant_ids: list of relevant document IDs (ground truth)
        k: cutoff

    Returns:
        Precision@k as a float. Returns 0.0 if k is 0.
    """
    ## YOUR CODE HERE
    raise NotImplementedError("Implement precision_at_k before running this cell.")


def recall_at_k(retrieved_ids: List[int], relevant_ids: List[int], k: int) -> float:
    """Compute recall@k.

    Args:
        retrieved_ids: list of retrieved document IDs (in ranked order)
        relevant_ids: list of relevant document IDs (ground truth)
        k: cutoff

    Returns:
        Recall@k as a float. Returns 0.0 if there are no relevant documents.
    """
    ## YOUR CODE HERE
    raise NotImplementedError("Implement recall_at_k before running this cell.")

In [ ]:
if __name__ == '__main__':
    # Sanity check for precision@k and recall@k
    assert abs(precision_at_k([1, 2, 3, 4, 5], [1, 3, 5, 7, 9], 5) - 0.6) < 1e-6
    assert abs(recall_at_k([1, 2, 3, 4, 5], [1, 3, 5, 7, 9], 5) - 0.6) < 1e-6
    assert abs(precision_at_k([1, 2, 3], [1, 3], 3) - 2/3) < 1e-6
    assert abs(recall_at_k([1, 2, 3], [1, 3], 3) - 1.0) < 1e-6
    print("Sanity checks passed!")

In [ ]:
if __name__ == '__main__':
    # Evaluate TF-IDF and BM25 on all evaluation queries
    def evaluate_retriever(retriever, eval_queries, k=5):
        """Evaluate a retriever on the evaluation queries.

        Args:
            retriever: retriever object with .retrieve(query, top_k) method
            eval_queries: dict of query_id -> {"query": str, "relevant_doc_ids": [int]}
            k: evaluation cutoff

        Returns:
            Dict with mean precision@k and mean recall@k
        """
        precisions, recalls = [], []
        for qid, qinfo in eval_queries.items():
            results = retriever.retrieve(qinfo["query"], top_k=k)
            retrieved_ids = [doc_id for doc_id, _ in results]
            p = precision_at_k(retrieved_ids, qinfo["relevant_doc_ids"], k)
            r = recall_at_k(retrieved_ids, qinfo["relevant_doc_ids"], k)
            precisions.append(p)
            recalls.append(r)
        return {
            "mean_precision@k": np.mean(precisions),
            "mean_recall@k": np.mean(recalls)
        }

    tfidf_results = evaluate_retriever(tfidf_retriever, EVAL_QUERIES, k=5)
    bm25_results = evaluate_retriever(bm25_retriever, EVAL_QUERIES, k=5)

    print("Sparse Retrieval Evaluation (k=5):")
    print(f"  TF-IDF  — P@5: {tfidf_results['mean_precision@k']:.4f}, R@5: {tfidf_results['mean_recall@k']:.4f}")
    print(f"  BM25    — P@5: {bm25_results['mean_precision@k']:.4f}, R@5: {bm25_results['mean_recall@k']:.4f}")

---
## Section 2: Dense Retrieval (20 points)

Now let's implement dense retrieval using pretrained sentence embeddings. This approach addresses the **vocabulary mismatch problem** (see **Lecture M10.2.1** and **Chapter 11, Section 11.3**).

We use the `sentence-transformers` library with the `all-MiniLM-L6-v2` model, a lightweight bi-encoder that maps sentences to 384-dimensional dense vectors.

### 2.1 Build Dense Retriever (10 points) [auto-checked]

Implement a dense retriever that:
1. Encodes all documents using a sentence-transformers model
2. At query time, encodes the query and computes cosine similarity with all document embeddings
3. Returns the top-k most similar documents

This implements the **bi-encoder architecture** from **Lecture M10.2.1**: encode queries and documents separately, then use dot product (or cosine similarity) for scoring.

**Model & docs:** we use the `all-MiniLM-L6-v2` bi-encoder from the `sentence-transformers` library. Reference: the [library docs (sbert.net)](https://www.sbert.net/) for the API (`SentenceTransformer`, `.encode(...)`) and the [model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) for details on this specific model. `model.encode(texts, normalize_embeddings=True)` returns unit-normalized vectors, so a dot product gives cosine similarity directly.

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
class DenseRetriever:
    """Dense retriever using sentence-transformers bi-encoder.

    Implements the bi-encoder architecture from Chapter 11, Section 11.3:
    - Encode documents offline
    - At query time, encode query and compute cosine similarity
    """

    def __init__(self, documents: List[Dict], model_name: str = 'all-MiniLM-L6-v2'):
        """Build the dense index.

        Args:
            documents: list of dicts with 'id', 'title', 'text' keys
            model_name: sentence-transformers model name
        """
        self.documents = documents

        ## YOUR CODE HERE
        # Import sentence_transformers HERE, inside __init__ (a "lazy" import) —
        # NOT at the top of the file — so the autograder can import Section 1
        # without loading torch.
        # 1. Load the sentence-transformers model
        # 2. Create text representations for each document (title + text)
        # 3. Encode all documents into dense vectors (use model.encode())
        # 4. Store the document embeddings as a numpy array
        #    Hint: model.encode() returns a numpy array of shape (n_docs, embedding_dim)

        self.model = None  # TODO
        self.doc_embeddings = None  # TODO: shape (n_docs, embedding_dim)

        raise NotImplementedError("Implement DenseRetriever.__init__ before running this cell.")

    def retrieve(self, query: str, top_k: int = 5) -> List[Tuple[int, float]]:
        """Retrieve top-k documents for a query.

        Args:
            query: query string
            top_k: number of documents to return

        Returns:
            List of (doc_id, score) tuples, sorted by score descending
        """
        ## YOUR CODE HERE
        # 1. Encode the query using the same model
        # 2. Compute cosine similarity between query embedding and all doc embeddings
        #    Hint: use numpy. Cosine similarity = dot product of normalized vectors.
        #    sentence-transformers returns normalized vectors by default.
        # 3. Return top-k (doc_id, score) pairs sorted by score descending
        raise NotImplementedError("Implement DenseRetriever.retrieve before running this cell.")

In [ ]:
# Skipped on `import` (only runs in the notebook). Lets the autograder
# import this file without downloading models or calling APIs (nothing here runs on import).
if __name__ == '__main__':
    # Build dense retriever (this will download the model on first run, ~80MB)
    print("Loading sentence-transformers model (first run downloads ~80MB)...")
    dense_retriever = DenseRetriever(DOCUMENTS)
    print(f"Document embeddings shape: {dense_retriever.doc_embeddings.shape}")

    # Test on a query
    results = dense_retriever.retrieve("How does semantic search work?", top_k=5)
    print("\nQuery: 'How does semantic search work?'")
    for doc_id, score in results:
        print(f"  [{doc_id}] {DOCUMENTS[doc_id]['title']}: {score:.4f}")

### 2.2 Evaluate and Compare (10 points) [manually graded]

Evaluate the dense retriever on the same queries as the sparse retrievers, then compare all three methods. **All 10 points are awarded through the written [WRITEUP] Answer 2.2 below** (the code here just produces the numbers you will analyze there).

In [ ]:
# Skipped on `import` (only runs in the notebook). Lets the autograder
# import this file without downloading models or calling APIs (nothing here runs on import).
if __name__ == '__main__':
    # Evaluate dense retriever
    dense_results = evaluate_retriever(dense_retriever, EVAL_QUERIES, k=5)

    print("Retrieval Evaluation (k=5):")
    print(f"  TF-IDF  — P@5: {tfidf_results['mean_precision@k']:.4f}, R@5: {tfidf_results['mean_recall@k']:.4f}")
    print(f"  BM25    — P@5: {bm25_results['mean_precision@k']:.4f}, R@5: {bm25_results['mean_recall@k']:.4f}")
    print(f"  Dense   — P@5: {dense_results['mean_precision@k']:.4f}, R@5: {dense_results['mean_recall@k']:.4f}")

In [ ]:
# Skipped on `import` (only runs in the notebook). Lets the autograder
# import this file without downloading models or calling APIs (nothing here runs on import).
if __name__ == '__main__':
    # Compare results on the vocabulary mismatch query (query 11)
    # NOTE: doc 34 (the relevant doc) actually shares the words "automobile",
    # "car", and "repair" with the query, so sparse methods can match it by exact
    # overlap here. Report what you observe — dense retrieval does not always win.
    vocab_mismatch_query = EVAL_QUERIES[11]["query"]
    print(f"Vocabulary mismatch query: '{vocab_mismatch_query}'")
    print(f"Relevant doc: {EVAL_QUERIES[11]['relevant_doc_ids']} ({DOCUMENTS[34]['title']})")

    print("\nTF-IDF results:")
    for doc_id, score in tfidf_retriever.retrieve(vocab_mismatch_query, top_k=3):
        print(f"  [{doc_id}] {DOCUMENTS[doc_id]['title']}: {score:.4f}")

    print("\nBM25 results:")
    for doc_id, score in bm25_retriever.retrieve(vocab_mismatch_query, top_k=3):
        print(f"  [{doc_id}] {DOCUMENTS[doc_id]['title']}: {score:.4f}")

    print("\nDense results:")
    for doc_id, score in dense_retriever.retrieve(vocab_mismatch_query, top_k=3):
        print(f"  [{doc_id}] {DOCUMENTS[doc_id]['title']}: {score:.4f}")

**[WRITEUP] Answer 2.2 (10 points):** Compare sparse and dense retrieval. Your answer should address:

1. Report precision@5 and recall@5 for all three methods (TF-IDF, BM25, Dense) in a table.
2. Which method performs best overall? Is the difference large or small?
3. Look at query 11 ("How do automobile repair guides help car owners?"), whose relevant document (doc 34) is the passage explaining the **vocabulary mismatch problem**. Report which method(s) actually retrieve doc 34 in their top-k, and explain the result based on the words the query and doc 34 do (or do not) share. Then, in general terms, explain why sparse methods (TF-IDF, BM25) can struggle on vocabulary mismatch and how dense retrieval is designed to help. *(See Lecture M10.2.1 and textbook §11.3 on the vocabulary mismatch problem. Note: dense retrieval does not always win in practice — report what you actually observe rather than what you expect.)*
4. Can you find a query where sparse retrieval outperforms dense retrieval? Why might this happen?

*(5-8 paragraphs. Reference specific queries and results from your experiments. Note: several eval queries have only 1-2 relevant documents, so a low precision@5 (≤ 0.4) is expected for those and is not a bug.)*

---
## Section 3: Building a RAG Pipeline (30 points)

Now let's build a complete **retrieval-augmented generation** pipeline. This section implements the retrieve-then-read architecture from **Lecture M10.4.2** and **Chapter 11, Section 11.4**.

The pipeline has four stages:
1. **Chunk** documents into smaller passages
2. **Index** chunks using dense embeddings
3. **Retrieve** relevant chunks for a query
4. **Generate** an answer using an LLM conditioned on retrieved chunks

### 3.1 Document Chunking (8 points) [auto-checked]

Implement a function that splits documents into overlapping chunks of a specified size. This connects to **Lecture M10.4.3** on chunking strategies.

Your chunker should:
- Split text into chunks of approximately `chunk_size` words
- Add `overlap` words of overlap between consecutive chunks
- Preserve the document source metadata (title, doc_id) for each chunk

**Worked example.** Chunking `"the quick brown fox jumps over the lazy dog"` (9 words) with `chunk_size=4, overlap=1` gives three chunks:
```
chunk 0: "the quick brown fox"
chunk 1: "fox jumps over the"     # 1-word overlap: "fox"
chunk 2: "the lazy dog"           # 1-word overlap: "the"; last chunk is shorter
```
Each step advances by `chunk_size - overlap = 3` words. Getting this exactly right matters — Sections 3.2–4 all build on your chunks.

In [ ]:
def chunk_documents(documents: List[Dict], chunk_size: int = 50,
                    overlap: int = 10) -> List[Dict]:
    """Split documents into overlapping chunks.

    Each chunk is a dict with:
        - 'chunk_id': int (unique across all chunks)
        - 'doc_id': int (source document ID)
        - 'title': str (source document title)
        - 'text': str (chunk text)

    Chunking strategy:
        - Split each document's text into words
        - Create chunks of chunk_size words with overlap words shared
          between consecutive chunks
        - The last chunk may be shorter than chunk_size

    Args:
        documents: list of dicts with 'id', 'title', 'text' keys
        chunk_size: target number of words per chunk
        overlap: number of overlapping words between consecutive chunks

    Returns:
        List of chunk dicts
    """
    ## YOUR CODE HERE
    # Caution: advance by max(chunk_size - overlap, 1) words per step. If you use
    # chunk_size - overlap directly and a caller passes overlap >= chunk_size, the
    # step becomes 0 and the loop never terminates (hangs the kernel).
    raise NotImplementedError("Implement chunk_documents before running this cell.")

In [ ]:
if __name__ == '__main__':
    # Test chunking with a small chunk size
    chunks_small = chunk_documents(DOCUMENTS, chunk_size=30, overlap=5)
    print(f"Chunk size=30, overlap=5: {len(chunks_small)} chunks")
    print(f"\nFirst 3 chunks:")
    for chunk in chunks_small[:3]:
        print(f"  Chunk {chunk['chunk_id']} (from doc {chunk['doc_id']}: {chunk['title']})")
        print(f"    {chunk['text'][:100]}...")

    # Test with larger chunks
    chunks_large = chunk_documents(DOCUMENTS, chunk_size=40, overlap=8)
    print(f"\nChunk size=40, overlap=8: {len(chunks_large)} chunks")

    # Sanity checks — chunking correctness impacts the rest of the assignment
    assert len(chunks_small) > len(DOCUMENTS), "Should have more chunks than documents"
    assert all({'chunk_id', 'doc_id', 'title', 'text'} <= set(ch) for ch in chunks_small), \
        "Each chunk must have chunk_id, doc_id, title, text"
    assert [ch['chunk_id'] for ch in chunks_small] == list(range(len(chunks_small))), \
        "chunk_id must be unique and sequential (0..n-1)"
    _ex = chunk_documents([{'id': 0, 'title': 'T',
                            'text': 'the quick brown fox jumps over the lazy dog'}],
                          chunk_size=4, overlap=1)
    assert [c['text'] for c in _ex] == ['the quick brown fox', 'fox jumps over the', 'the lazy dog'], \
        f"Overlap/size wrong on the worked example, got {[c['text'] for c in _ex]}"
    print("\nSanity checks passed!")

### 3.2 Chunk Indexing (7 points) [auto-checked]

Build a dense index over the chunks using sentence-transformers embeddings and numpy for similarity search. No external vector database needed — for our small corpus, brute-force cosine search is fast enough.

In [ ]:
class ChunkIndex:
    """Dense index over document chunks using sentence-transformers.

    Uses cosine similarity with numpy for retrieval (no external vector DB).
    """

    def __init__(self, chunks: List[Dict], model: "SentenceTransformer"):
        """Build the index.

        Args:
            chunks: list of chunk dicts with 'chunk_id', 'doc_id', 'title', 'text'
            model: sentence-transformers model for encoding
        """
        self.chunks = chunks
        self.model = model

        ## YOUR CODE HERE
        # 1. Create text representations for each chunk
        #    (include the title for context: f"{chunk['title']}: {chunk['text']}")
        # 2. Encode all chunks into dense vectors
        # 3. Store as self.chunk_embeddings (numpy array)

        self.chunk_embeddings = None  # TODO: shape (n_chunks, embedding_dim)

        raise NotImplementedError("Implement ChunkIndex.__init__ before running this cell.")

    def search(self, query: str, top_k: int = 3) -> List[Dict]:
        """Search for the most relevant chunks.

        Args:
            query: query string
            top_k: number of chunks to return

        Returns:
            List of dicts, each with keys:
                'chunk': the chunk dict
                'score': cosine similarity score
        """
        ## YOUR CODE HERE
        # 1. Encode the query
        # 2. Compute cosine similarity with all chunk embeddings
        # 3. Return top-k results with chunk info and scores
        raise NotImplementedError("Implement ChunkIndex.search before running this cell.")

In [ ]:
# Skipped on `import` (only runs in the notebook). Lets the autograder
# import this file without downloading models or calling APIs (nothing here runs on import).
if __name__ == '__main__':
    # Build chunk index with small chunks
    # Reuse the model from the DenseRetriever
    model = dense_retriever.model

    chunks = chunk_documents(DOCUMENTS, chunk_size=50, overlap=10)
    chunk_index = ChunkIndex(chunks, model)
    print(f"Indexed {len(chunks)} chunks, embeddings shape: {chunk_index.chunk_embeddings.shape}")

    # Test search
    results = chunk_index.search("How does RAG reduce hallucination?", top_k=3)
    print("\nQuery: 'How does RAG reduce hallucination?'")
    for r in results:
        print(f"  Score: {r['score']:.4f} | Doc: {r['chunk']['title']} | Text: {r['chunk']['text'][:80]}...")

### 3.3 RAG Generation (15 points) [structure auto-checked · 15 pts by TA from notebook output]

Build the **reader** component: given retrieved chunks and a question, construct a prompt and call the Anthropic API to generate an answer.

This implements the RAG prompt construction from **Lecture M10.4.2** and **Chapter 11, Section 11.4**.

In [ ]:
########## PROVIDED — provider-agnostic LLM helper (Anthropic OR OpenAI) ##########
# You only need ONE account for the whole course. Default is Anthropic; to use
# OpenAI instead, set USE_OPENAI = True and export OPENAI_API_KEY.
#   Anthropic:  export ANTHROPIC_API_KEY="..."   (default)
#   OpenAI:     export OPENAI_API_KEY="..."  and set USE_OPENAI = True below
USE_OPENAI = False

def llm_generate(prompt: str, model: str = None,
                 temperature: float = 0.0, max_tokens: int = 800) -> str:
    """Call the LLM and return the generated text. Works with either provider."""
    if USE_OPENAI:
        from openai import OpenAI
        client = OpenAI()
        resp = client.chat.completions.create(
            model=model or "gpt-4o-mini",
            max_tokens=max_tokens, temperature=temperature,
            messages=[{"role": "user", "content": prompt}])
        return resp.choices[0].message.content
    else:
        import anthropic
        client = anthropic.Anthropic()
        resp = client.messages.create(
            model=model or "claude-sonnet-4-6",
            max_tokens=max_tokens, temperature=temperature,
            messages=[{"role": "user", "content": prompt}])
        return resp.content[0].text



def build_rag_prompt(question: str, retrieved_chunks: List[Dict]) -> str:
    """Construct a RAG prompt from a question and retrieved chunks.

    The prompt should include:
    1. An instruction to answer based ONLY on the provided passages
    2. The retrieved passages, numbered and with source titles
    3. An instruction to cite which passage(s) support the answer
    4. An instruction to say "I don't have enough information" if the passages
       don't contain the answer
    5. The question

    Args:
        question: the user's question
        retrieved_chunks: list of dicts from ChunkIndex.search()
            each with 'chunk' (dict with 'title', 'text') and 'score'

    Returns:
        The formatted prompt string
    """
    ## YOUR CODE HERE
    raise NotImplementedError("Implement build_rag_prompt before running this cell.")


def generate_answer(question: str, retrieved_chunks: List[Dict]) -> str:
    """Generate an answer using RAG.

    Args:
        question: the user's question
        retrieved_chunks: list of dicts from ChunkIndex.search()

    Returns:
        Generated answer string
    """
    ## YOUR CODE HERE
    # 1. Build the prompt with build_rag_prompt()
    # 2. Return llm_generate(prompt) — provider-agnostic (Anthropic or OpenAI),
    #    temperature=0 by default for reproducibility.
    raise NotImplementedError("Implement generate_answer before running this cell.")

In [ ]:
# Skipped on `import` (only runs in the notebook). Lets the autograder
# import this file without downloading models or calling APIs (nothing here runs on import).
if __name__ == '__main__':
    # Test the RAG pipeline end-to-end
    test_question = RAG_QUESTIONS[0]  # "What is retrieval-augmented generation..."
    print(f"Question: {test_question}")

    # Retrieve
    retrieved = chunk_index.search(test_question, top_k=3)
    print(f"\nRetrieved {len(retrieved)} chunks:")
    for i, r in enumerate(retrieved):
        print(f"  [{i+1}] {r['chunk']['title']} (score: {r['score']:.4f})")

    # Show the prompt
    prompt = build_rag_prompt(test_question, retrieved)
    print(f"\n--- RAG Prompt ---")
    print(prompt)
    print(f"--- End Prompt ---")

    # Generate answer
    answer = generate_answer(test_question, retrieved)
    print(f"\n--- Generated Answer ---")
    print(answer)
    print(f"--- End Answer ---")

In [ ]:
# Skipped on `import` (only runs in the notebook). Lets the autograder
# import this file without downloading models or calling APIs (nothing here runs on import).
if __name__ == '__main__':
    # Run RAG on all questions and store results
    rag_results = []
    for question in RAG_QUESTIONS:
        retrieved = chunk_index.search(question, top_k=3)
        answer = generate_answer(question, retrieved)
        rag_results.append({
            "question": question,
            "retrieved_chunks": retrieved,
            "answer": answer
        })
        print(f"Q: {question}")
        print(f"A: {answer[:200]}...\n")

    print(f"Generated answers for {len(rag_results)} questions.")

---
## Section 4: RAG Evaluation & Analysis (18 points)

Now let's evaluate the RAG system. We'll assess both **retrieval quality** and **generation faithfulness**, and compare different chunking strategies.

This connects to **Lecture M10.4.2** on RAG evaluation and **Lecture M10.4.3** on chunking tradeoffs.

### 4.1 Retrieval Quality Analysis (5 points) [WRITEUP — manually graded]

**Answer in your `hw08_writeup.pdf`.** For each RAG question, examine the retrieved chunks. Are they relevant? Do they contain the information needed to answer the question? **Cite the specific retrieved chunks** (by title / passage number) that support — or fail to support — each question.

In [ ]:
# Skipped on `import` (only runs in the notebook). Lets the autograder
# import this file without downloading models or calling APIs (nothing here runs on import).
if __name__ == '__main__':
    # Display retrieved chunks for each question
    for i, result in enumerate(rag_results):
        print(f"\n{'='*80}")
        print(f"Q{i+1}: {result['question']}")
        print(f"{'='*80}")
        for j, chunk_result in enumerate(result['retrieved_chunks']):
            chunk = chunk_result['chunk']
            print(f"  Chunk {j+1} (score: {chunk_result['score']:.4f}) — {chunk['title']}")
            print(f"    {chunk['text'][:150]}...")
        print(f"\n  Answer: {result['answer'][:200]}...")

### 4.2 Chunking Strategy Comparison (5 points) [WRITEUP — manually graded]

Compare at least two different chunk sizes. Build indexes with different configurations and evaluate retrieval quality. **Put your comparison (table + short analysis) in your `hw08_writeup.pdf`.** In your printed comparison, address: which chunk size retrieves chunks that actually contain the answer, and the precision-vs-context tradeoff (small chunks are more focused but may split the answer; large chunks keep context but add noise).

### Suggested Table for Section 4.2: Chunking Strategy Comparison

| Chunking Strategy         | Question                                                          | Relevant Chunks Retrieved? (Yes/No/Partial) | Notes                                                                                                    |
|---------------------------|-------------------------------------------------------------------|---------------------------------------------|----------------------------------------------------------------------------------------------------------|
| `chunk_size=20, overlap=5`| What is retrieval-augmented generation and why does it help with hallucination? |                                             | *(e.g., 'Retrieved relevant chunks 11, 36. Answerable from context.')*                                  |
| `chunk_size=50, overlap=10`| What is retrieval-augmented generation and why does it help with hallucination? |                                             | *(e.g., 'Retrieved less precise chunks, some irrelevant info in top-3. Still answerable.')*                |
| `chunk_size=20, overlap=5`| How does BM25 differ from basic TF-IDF?                           |                                             |                                                                                                          |
| `chunk_size=50, overlap=10`| How does BM25 differ from basic TF-IDF?                           |                                             |                                                                                                          |
| ...                       | ...                                                               |                                             |                                                                                                          |

After filling this table, you can use it to inform your analysis in the writeup, discussing which chunk size retrieves chunks that actually contain the answer and the precision-vs-context tradeoff.

In [ ]:
# Skipped on `import` (only runs in the notebook). Lets the autograder
# import this file without downloading models or calling APIs (nothing here runs on import).
if __name__ == '__main__':
    ## YOUR CODE HERE
    # 1. Create chunks with at least two different chunk_size values
    #    (e.g., chunk_size=20/overlap=5 vs chunk_size=50/overlap=10)
    # 2. Build a ChunkIndex for each
    # 3. For each RAG question, retrieve top-3 chunks from each index
    # 4. Compare: which chunk size retrieves more relevant content?
    #    You can do this qualitatively (inspect the results) or
    #    quantitatively (if you define relevance labels for chunks)

    raise NotImplementedError("Implement the 4.2 chunk-size comparison before running this cell.")

### 4.3 Generation Faithfulness (5 points) [WRITEUP — manually graded]

**Answer in your `hw08_writeup.pdf`.** Evaluate whether the generated answers are **faithful** to the retrieved passages — that is, whether the claims in the answer are supported by the retrieved text.

For each of the 5 RAG answers:
1. Identify the key claims in the answer
2. Check whether each claim is supported by the retrieved passages
3. Flag any claims that appear to be hallucinated (not in the passages)

**Cite the specific passage(s)** that support each claim. For each answer, state your faithfulness score: *"N out of M claims are supported by the retrieved passages."* If an answer has no unsupported claims, state *"no observed hallucinations (M/M claims supported)"* instead.

After evaluating all 5 answers, write a brief overall summary: which answers were most/least faithful, and whether you observed any patterns in where hallucinations occurred.

### 4.4 Systematic Chunking Comparison (3 points) [WRITEUP — manually graded]

Compare at least **three** chunking strategies on the same set of queries using the same retrieval metrics (precision and recall of the retrieved chunks, judged qualitatively or quantitatively):

| Strategy | Description |
|----------|-------------|
| Fixed-size 20 words | `chunk_documents(DOCUMENTS, chunk_size=20, overlap=5)` |
| Fixed-size 40 words | `chunk_documents(DOCUMENTS, chunk_size=40, overlap=8)` |
| Paragraph-based | Each document is a single "chunk" (no splitting) |

For each strategy:
1. Build a `ChunkIndex` and retrieve the top-3 chunks for every question in `RAG_QUESTIONS`.
2. Record which source documents appear in the retrieved chunks and how relevant the chunk text is.

**[WRITEUP] (3 pts):** Create a comparison table showing, for each strategy and each query, whether the top-3 retrieved chunks contain the information needed to answer the question. Then write 2-3 paragraphs explaining:
- Which strategy works best overall and why.
- How chunk size affects the precision/context tradeoff discussed in Lecture M10.4.3.
- Whether overlap helps, and in what situations paragraph-based chunking might be preferable.

### Sample Evaluation Table for Section 4.4: Systematic Chunking Comparison

| Chunking Strategy         | Question                                                          | Top-3 Chunks Contain Answer? (Yes/No/Partial) |
|---------------------------|-------------------------------------------------------------------|-----------------------------------------------|
| Fixed-size 20 words       | What is retrieval-augmented generation and why does it help with hallucination? |                                               |                                    |
| Fixed-size 40 words       | What is retrieval-augmented generation and why does it help with hallucination? |                                               |
| Paragraph-based           | What is retrieval-augmented generation and why does it help with hallucination? |                                               |



In [ ]:
# Skipped on `import` (only runs in the notebook). Lets the autograder
# import this file without downloading models or calling APIs (nothing here runs on import).
if __name__ == '__main__':
    ## YOUR CODE HERE (3 points)
    # 1. Build three chunk sets:
    #    chunks_20 = chunk_documents(DOCUMENTS, chunk_size=20, overlap=5)
    #    chunks_40 = chunk_documents(DOCUMENTS, chunk_size=40, overlap=8)
    #    chunks_para = [{"chunk_id": doc["id"], "doc_id": doc["id"],
    #                    "title": doc["title"], "text": doc["text"]} for doc in DOCUMENTS]
    #
    # 2. Build a ChunkIndex for each (reuse `model` from earlier):
    #    index_20 = ChunkIndex(chunks_20, model)
    #    index_40 = ChunkIndex(chunks_40, model)
    #    index_para = ChunkIndex(chunks_para, model)
    #
    # 3. For each RAG question, retrieve top-3 from each index and print results side by side.
    # 4. Summarize in a table: for each (strategy, question), note whether the retrieved
    #    chunks contain the answer.

    raise NotImplementedError("Implement the 4.4 systematic chunking comparison before running this cell.")

---
## Section 5: Reflection (12 points) [WRITEUP]

These questions ask you to synthesize concepts from across Module 10.

**[WRITEUP] Answer 5.1 (4 points): When Does RAG Help vs. Hurt?**

Retrieval-augmented generation is not always better than prompting an LLM directly.

a) Describe **two specific scenarios** where RAG would significantly improve answer quality compared to a standalone LLM. For each, explain *why* retrieval helps.

b) Describe **two specific scenarios** where RAG might hurt performance or add unnecessary complexity. For each, explain *why* retrieval doesn't help (or actively harms).

*(Refer to Lecture M10.4.2 on when RAG helps vs. hurts, and the textbook's discussion of hallucination in Chapter 11.)*

**[WRITEUP] Answer 5.2 (4 points): Sparse vs. Dense Retrieval Tradeoffs**

a) In your experiments, you compared sparse (TF-IDF, BM25) and dense (sentence-transformers) retrieval. Beyond what you observed, discuss the **practical tradeoffs** between these approaches in a production system. Consider:
- Computational cost (indexing and query time)
- Storage requirements
- Handling of rare terms and proper nouns
- Need for labeled training data

Address **at least 3** of the 4 dimensions above.

b) Many production systems use **hybrid retrieval** (combining sparse and dense). Why might this work better than either approach alone? How would you combine their scores?

*(Refer to Lecture M10.2.1 on dense retrieval and the textbook discussion of hybrid approaches in Chapter 11.)*

**[WRITEUP] Answer 5.3 (4 points): Hallucination in RAG Systems**

In Section 4.3, you evaluated whether RAG-generated answers were faithful to the retrieved passages.

a) Even with relevant passages retrieved, LLMs can still hallucinate in RAG systems. Describe **at least two mechanisms** by which hallucination can occur in a RAG pipeline — i.e., what can go wrong *between* retrieval and generation. Think about how the model uses (or fails to use) the retrieved passages, and what happens when the passages are only partially relevant.

b) Suggest **two concrete strategies** to reduce hallucination in RAG systems. For each, explain how it addresses the failure mode and what tradeoffs it involves.

*(Connect to the discussion of hallucination in Lecture M10.4.1. No prior module beyond Module 10 is required.)*

---
## Submission Checklist

Before submitting to Gradescope, verify:

- [ ] All code cells run without errors (Kernel -> Restart & Run All)
- [ ] `hw08_writeup.pdf` contains answers to all **[WRITEUP]** questions:
  - Answer 2.2 (sparse vs. dense comparison — 10 pts)
  - Answer 4.1 (retrieval quality analysis — 5 pts)
  - Answer 4.2 (chunking strategy comparison — 5 pts)
  - Answer 4.3 (generation faithfulness — 5 pts)
  - Answer 4.4 (systematic chunking comparison — 3 pts)
  - Answer 5.1 (when RAG helps vs. hurts — 4 pts)
  - Answer 5.2 (sparse vs. dense tradeoffs — 4 pts)
  - Answer 5.3 (hallucination in RAG — 4 pts)
- [ ] All files are named exactly: `hw08.py`, `hw08.ipynb`, `hw08_writeup.pdf`